# PRATYAKSHA · Step 4 — R4 (what errors are left?) and R5 (fine-tuning)

**What Step 3 showed:** with computed options, the 2B model is ~90% right and never wrong automatically,
but it sends ~75% of lines to a person. The 0.8B model is at ~63%.
**This notebook:** (R4) count which errors fine-tuning *could* fix, then (R5) teach the models on
months 1–4 of the business's own past reconciliations, and test on month 5 as before.

Runtime **T4 GPU**. Upload `pratyaksha_step4.zip` to *Colab Notebooks* first.
**Long notebook (about 2–2.5 h).** You can stop after any section: results and trained models are saved
to Drive. To continue in a new session, run cells 1–4 again, then jump to where you stopped.

## 1. Connect Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Unpack Step 4 code
**Why:** adds the fine-tuning code; earlier results stay.

In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step4.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!ls pratyaksha/finetune

/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Colab Notebooks/pratyaksha
build_sft.py  __init__.py  train.py


## 3. Install
**Why:** `peft` adds LoRA; `bitsandbytes` gives the 8-bit optimiser that lets full fine-tuning fit on a T4.

In [3]:
!pip install -q -U transformers accelerate sentence-transformers rank-bm25 pytest peft bitsandbytes

## 4. Data and tests
**Expect:** `29 passed`. New tests check that training data comes only from months 1–4 and that every target answer is correct.

In [4]:
!python -m pratyaksha.world.generate --out data/wwi_in
!python -m pytest -q

{"customers": 60, "suppliers": 8, "invoices": 1361, "bank_entries": 997, "remittances": 501}
.............................                                            [100%]
29 passed in 9.34s


## 5. R4 — what errors are left after R3? (no GPU work, reads Step 3 results)
**Why:** fine-tune only if the remaining errors are the kind training can fix (our report's rule).
- *correct + review* = right but sent to a person → training can teach it when **not** to ask
- *wrong, right answer WAS offered* = picked the wrong option → training can fix
- *wrong, right answer NOT offered* = our options missed it → training **cannot** fix

In [5]:
!python scripts/error_analysis.py --model R3_Qwen3.5-2B
!python scripts/error_analysis.py --model R3_Qwen3.5-0.8B


== R3_Qwen3.5-2B · clean · n=177
  correct+auto                         46   26.0%
  correct+review                      113   63.8%
  wrong, right answer WAS offered       8    4.5%
  wrong, right answer NOT offered       9    5.1%
  unusable output                       1    0.6%
  -> fine-tuning could fix up to 122 lines (68.9%); options/retrieval must fix 9
  wrong choices by category: {'short_payment': 3, 'unknown_party': 1, 'duplicate_credit': 2, 'tds_deduction': 1, 'bank_charges': 1}

== R3_Qwen3.5-2B · stress · n=177
  correct+auto                         35   19.8%
  correct+review                      122   68.9%
  wrong, right answer WAS offered       9    5.1%
  wrong, right answer NOT offered       9    5.1%
  unusable output                       2    1.1%
  -> fine-tuning could fix up to 133 lines (75.1%); options/retrieval must fix 9
  wrong choices by category: {'short_payment': 3, 'unknown_party': 1, 'duplicate_credit': 3, 'bank_charges': 1, 'tds_deduction': 1}

== R

## 6. Re-run the 4B model with "thinking" switched off
**Why:** in Step 3 the 4B model wrote a long "Thinking Process" and was cut off on every line.
The code now turns thinking off. This checks the fix (clean lines only, ~8 min).
**Expect:** far fewer than 177 unusable answers.

In [6]:
!python -m pratyaksha.harness.runner --system r3 --split dev --condition clean --cost-mode gpu \
    --opt model=Qwen/Qwen3.5-4B --opt batch=4 --tag nothink

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.42MB/s]
chat_template.jinja: 100% 7.76k/7.76k [00:00<00:00, 23.4MB/s]
config.json: 100% 3.16k/3.16k [00:00<00:00, 7.35MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 34.4MB/s]

tokenizer.json: downloading bytes:   0% 0.00/12.8M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 5.68M/5.68M [00:00<00:00, 6.96MB/s,  559kB/s  ]
tokenizer.json: reconstructing file: 100% 12.8M/12.8M [00:00<00:00, 15.7MB/s, 1.25MB/s  ]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.96MB/s]
model.safetensors.index.json: 100% 76.2k/76.2k [00:00<00:00, 170MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.33G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  14% 1.34G/9.32G [00:20<02:36, 50.9MB/s, 80.6MB/s  ]
Reconstructing (incomplete total...):  63% 5.85G/9.32G [01:25<01:18, 44.1MB/s,

## 7. Build the training data (months 1–4 only)
**What:** for each past bank line (clean and messy), the exact R3 prompt plus the correct short answer,
e.g. `{"option": "M2", "needs_review": false}`.
**Why:** the business's own history is the teacher; month 5 (dev) and month 6 (test) are never used.
**Expect:** about 1,270 examples.

In [7]:
!python -m pratyaksha.finetune.build_sft --out data/sft/train.jsonl

Loading weights: 100% 393/393 [00:00<00:00, 4123.21it/s]
wrote 1270 examples to data/sft/train.jsonl
{
 "offered": 1124,
 "review=False": 1026,
 "unknown": 22,
 "review=True": 244,
 "not_offered": 102,
 "duplicate": 22
}


## 8. Dry run (5 training steps)
**Why:** catches memory or setup errors in 2 minutes instead of 20. **Expect:** a few loss values and "saved to".

In [8]:
!python -m pratyaksha.finetune.train --model Qwen/Qwen3.5-0.8B --mode lora --out models/dryrun --max-steps 5

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.53MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 22.7MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 11.1MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 48.7MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 2.51MB/s]
model.safetensors.index.json: 100% 50.9k/50.9k [00:00<00:00, 133MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:  10% 181M/1.75G [00:01<00:05, 261MB/s, 13.9MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  18% 321M/1.75G [00:01<00:05, 258MB/s, 28.4MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  49% 862M/1.75G [00:04<00:04, 210MB/s, 67.9MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  52% 908M/1.75G [00:04<00:03, 213MB/s, 70.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  56% 986M/1.75G [00:04<00:04, 184MB/s, 74.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…

## 9. R5a — LoRA fine-tune of the 0.8B model (~20–30 min), then test it
**What:** freeze the model; train small add-on layers (well under 5% of the weights).
**Why:** the main question of the project — how far can the smallest model get with training?
**Watch:** the loss should fall (roughly from >1 to <0.2).

In [9]:
!python -m pratyaksha.finetune.train --model Qwen/Qwen3.5-0.8B --mode lora --out models/r5_lora_0.8b
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r5 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-0.8B --opt adapter=models/r5_lora_0.8b --opt batch=8

Loading weights: 100% 473/473 [00:01<00:00, 315.28it/s]
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3204: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
examples 1270 (truncated 0) | trainable 10,822,656 of 863,808,576 (1.25%) | steps 318 | lr 0.0002
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` f

## 10. R5b — LoRA fine-tune of the 2B model (~40–60 min), then test it
**Why:** the 2B model was already accurate but over-cautious. Does training teach it when it is safe
to act alone, without making automatic mistakes?

In [ ]:
!python -m pratyaksha.finetune.train --model Qwen/Qwen3.5-2B --mode lora --out models/r5_lora_2b
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r5 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt adapter=models/r5_lora_2b --opt batch=8

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.46MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 13.0MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 8.39MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 44.2MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 2.36MB/s]
model.safetensors.index.json: 100% 64.5k/64.5k [00:00<00:00, 126MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:   5% 217M/4.55G [00:01<00:16, 270MB/s, 17.6MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:   7% 309M/4.55G [00:01<00:21, 201MB/s, 26.7MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  15% 701M/4.55G [00:03<00:24, 158MB/s, 57.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  31% 1.39G/4.55G [00:07<00:18, 168MB/s, 92.9MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  34% 1.56G/4.55G [00:09<00:20, 148MB/s, 91.1MB/s  ]
model.safetensors-00001-of-00001.safeten

In [5]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!ls

!python -m pratyaksha.harness.runner --system r5 --split dev --condition stress --cost-mode gpu \
    --opt model=Qwen/Qwen3.5-2B --opt adapter=models/r5_lora_2b --opt batch=8

import json, glob
run = sorted(glob.glob("runs/*R5_r5_lora_2b_dev_clean/predictions.jsonl"))[-1]
for r in [json.loads(l) for l in open(run)][:3]:
    print("-----"); print(repr(r["pred"]["raw_output"][:400]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/pratyaksha
data  models	 pratyaksha  requirements.txt  scripts
docs  notebooks  README.md   runs	       tests
Loading weights: 100% 617/617 [00:04<00:00, 141.78it/s]
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
config.json: 100% 795/795 [00:00<00:00, 4.65MB/s]

model.safetensors: downloading bytes:   9% 197M/2.27G [00:01<00:10, 203MB/s, 17.8MB/s  ]
model.safetensors: downloading bytes:  16% 367M/2.27G [00:02<00:07, 243MB/s, 30.4MB/s  ]
model.safetensors: downloading bytes:  22% 507M/2.27G [00:02<00:10, 167MB/s, 41.4MB/s  ]
model.safetensors: downlo

In [6]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step4b.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!python -m pratyaksha.harness.runner --system r5 --split dev --condition clean --cost-mode gpu \
    --opt model=Qwen/Qwen3.5-2B --opt adapter=models/r5_lora_2b --opt batch=8 --tag stopfix

/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Colab Notebooks/pratyaksha
Loading weights: 100% 617/617 [00:07<00:00, 87.79it/s] 
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
Loading weights: 100% 393/393 [00:00<00:00, 1612.48it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causa

## 11. (Optional) R5c — FULL fine-tune of the 0.8B model (~30–45 min), then test it
**Why:** LoRA vs full fine-tuning is in the mid-sem plan. Full training changes every weight.
**Note:** saves ~3.5 GB to Drive. Skip if Drive space is low or time is short.

In [7]:
!python -m pratyaksha.finetune.train --model Qwen/Qwen3.5-0.8B --mode full --out models/r5_full_0.8b
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r5 --split dev --condition {cond} --cost-mode gpu \
        --opt model=models/r5_full_0.8b --opt batch=8

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.79MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 21.0MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 2.13MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 49.5MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.51MB/s]
model.safetensors.index.json: 100% 50.9k/50.9k [00:00<00:00, 99.1MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:  13% 220M/1.75G [00:01<00:07, 214MB/s, 18.5MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  18% 316M/1.75G [00:02<00:10, 143MB/s, 27.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  22% 386M/1.75G [00:02<00:08, 162MB/s, 32.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  31% 540M/1.75G [00:03<00:06, 197MB/s, 43.3MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  34% 594M/1.75G [00:03<00:05, 230MB/s, 46.3MB/s  ]
model.safetensors-00001-of-00001.safeten(

## 12. Compare everything
**Why:** this is the Step 4 result: R3 vs R5, model size, LoRA vs full, against the no-AI control.

In [8]:
!python scripts/compare.py --split dev

system                      cond        n     acc    auto  autoErr   p50 s  party@k  tok_in parseErr   bar
R3_control_first_option     clean     177   89.8%   82.5%     0.0%    0.10    95.9%     339        0  FAIL
R3_control_first_option     stress    177   88.1%   80.8%     0.0%    0.11    95.3%     341        0  FAIL
R1_Qwen3.5-2B               clean     177   45.8%   81.9%    45.5%    1.92    90.6%    1929       22  FAIL
R1_Qwen3.5-2B               stress    177   40.7%   80.8%    50.3%    1.95    88.3%    1907       21  FAIL
R3_Qwen3.5-0.8B             clean     177   63.8%   57.1%     0.0%    0.75    95.9%     526        4  FAIL
R3_Qwen3.5-0.8B             stress    177   61.6%   55.4%     0.0%    0.78    95.3%     527        6  FAIL
R3_Qwen3.5-2B               clean     177   89.8%   26.0%     0.0%    0.73    95.9%     526        1  FAIL
R3_Qwen3.5-2B               stress    177   88.7%   19.8%     0.0%    0.76    95.3%     527        2  FAIL
R3_Qwen3.5-4B               clean    

## 13. What is still wrong after fine-tuning?
**Why:** shows whether training removed over-caution and wrong choices, and what is left for R6.

In [9]:
!python scripts/error_analysis.py --model R5_r5_lora_0.8b
!python scripts/error_analysis.py --model R5_r5_lora_2b
!python scripts/error_analysis.py --model R5_r5_full_0.8b


== R5_r5_lora_0.8b · clean · n=177
  correct+auto                        146   82.5%
  correct+review                       20   11.3%
  wrong, right answer WAS offered       1    0.6%
  wrong, right answer NOT offered      10    5.6%
  unusable output                       0    0.0%
  -> fine-tuning could fix up to 21 lines (11.9%); options/retrieval must fix 10
  wrong choices by category: {'unknown_party': 1}

== R5_r5_lora_0.8b · stress · n=177
  correct+auto                        145   81.9%
  correct+review                       20   11.3%
  wrong, right answer WAS offered       1    0.6%
  wrong, right answer NOT offered      11    6.2%
  unusable output                       0    0.0%
  -> fine-tuning could fix up to 21 lines (11.9%); options/retrieval must fix 11
  wrong choices by category: {'unknown_party': 1}

== R5_r5_lora_2b · clean · n=177
  correct+auto                        145   81.9%
  correct+review                       22   12.4%
  wrong, right answer WAS offer

## 14. Training cost
**Why:** fine-tuning is a one-time cost; we report it next to the per-task cost.

In [10]:
import json, glob
for f in sorted(glob.glob("models/r5_*/train_log.json")):
    d = json.load(open(f))
    print(f"{f.split('/')[1]:<16} {d['mode']:<5} {d['train_seconds']/60:6.1f} min  steps {d['steps']:<4} "
          f"trainable {d['trainable_params']:,} of {d['total_params']:,}  loss {d['loss'][0]['loss']} -> {d['loss'][-1]['loss']}")

r5_full_0.8b     full    42.4 min  steps 318  trainable 852,985,920 of 852,985,920  loss 0.9534 -> 0.0159
r5_lora_0.8b     lora    54.5 min  steps 318  trainable 10,822,656 of 863,808,576  loss 0.9537 -> 0.006
r5_lora_2b       lora    53.0 min  steps 318  trainable 16,819,200 of 2,230,060,864  loss 0.8959 -> 0.0


## 15. Send back
*File → Download → .ipynb* and send it.